# Part 1: two-speaker separation on Libri2Mix, fine-tuning SR-CorrNet

**Before running:** Settings, Accelerator **GPU T4 x2 or P100**, Internet **On**. No dataset upload is needed:
the data cell downloads LibriSpeech (`train-clean-100`, `dev-clean`, `test-clean`) and the official LibriMix
metadata, and the scripts build Libri2Mix `min` / `mix_clean` mixtures on the fly at 8 kHz. If a LibriSpeech or
generated Libri2Mix dataset is attached under *Add Input*, it is found and used instead of downloading.

Systems compared on the full Libri2Mix test set (8 kHz, `mix_clean`):

| name | what |
|---|---|
| `zeroshot` | official SR-CorrNet checkpoint (trained on WSJ0-2mix), no training |
| `ft_full` | same checkpoint fine-tuned end-to-end on Libri2Mix train-100 |
| `ft_frozen` | fine-tuned with the separation encoder frozen (ablation) |
| `blstm` | own BLSTM mask baseline trained from scratch |

To fit Kaggle's 12 h session, use *Save Version, Save & Run All (Commit)* so it runs in the background, and set the
hour budgets in the next cell. Each training run saves `last.pt` every epoch; set `RESUME = True` to continue.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation on 16 GB GPUs
# Settings
LS_DIR, META_DIR = "/tmp/librispeech", "/tmp/LibriMix"   # download targets (not saved as output)
DATA_ROOT = f"/kaggle/input,{LS_DIR},{META_DIR}/metadata"   # all searched; comma-separated
QUICK = False                 # True = 10-minute pipeline check on tiny subsets
# Defaults fit about 3 h on a T4 (estimate). SR-CorrNet costs ~365 GFLOPs per second of audio,
# so the full 3000-utterance test set would take over 1 h per system; we use a fixed random 500.
FT_HOURS, FROZEN_HOURS, BLSTM_HOURS = 1.25, 0.5, 0.5   # wall-clock training budgets
RUN_FROZEN, RUN_BLSTM = False, True   # frozen-encoder ablation adds ~50 min
EVAL_EXTRA = "--perceptual"  # STOI + PESQ; add "--sdr" for BSS-eval SDR (slow)
RESUME = False
# Without gradient checkpointing SR-CorrNet training stores ~4.2 GiB per audio-second, so even one 4 s clip
# (~17 GiB) overflows a 16 GB T4/P100. --grad_ckpt recomputes blocks in backward: same gradients, ~0.6 GiB stored.
FT_BATCH = "--batch_size 1 --accum 4 --segment 4 --grad_ckpt"

if QUICK:
    FT_HOURS = FROZEN_HOURS = BLSTM_HOURS = 0.05
    EVAL_N = "--max_utts 40"
    STEPS = "--steps_per_epoch 40 --val_items 40"
else:
    EVAL_N = "--max_utts 500"
    STEPS = "--steps_per_epoch 150 --val_items 200"
RES = "--resume" if RESUME else ""
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# official SR-CorrNet code, pinned to the commit this notebook was tested with (MIT licence)
!pip install -q "git+https://github.com/dmlguq456/SR_CorrNet_SS@7340365b9cc9a021bf7d400f52fce4b88593b67a" huggingface_hub mir_eval pystoi pesq

## Data: LibriSpeech + LibriMix metadata
About 7 GB from openslr.org, usually 5–15 minutes. Skipped for any part already present
(re-runs in the same session, or a LibriSpeech dataset attached under *Add Input*).

In [ ]:
import glob, os, subprocess
PARTS = ["dev-clean", "test-clean", "train-clean-100"]
MIRRORS = ["https://openslr.elda.org/resources/12", "https://www.openslr.org/resources/12",
           "https://us.openslr.org/resources/12"]

def present(part):
    pats = [f"{LS_DIR}/**/{part}/[0-9]*"] + [f"/kaggle/input/{'*/' * d}{part}/[0-9]*" for d in range(1, 5)]
    return any(glob.glob(p, recursive=True) for p in pats)

def fetch(part):
    for m in MIRRORS:  # stream straight into tar: no tarball kept on disk
        cmd = f"curl -fsSL --retry 5 --retry-delay 5 {m}/{part}.tar.gz | tar xz -C {LS_DIR}"
        if subprocess.run(["bash", "-o", "pipefail", "-c", cmd]).returncode == 0:
            return f"{part}: ok from {m}"
    return f"{part}: FAILED on all mirrors"

os.makedirs(LS_DIR, exist_ok=True)
todo = [p for p in PARTS if not present(p)]
print("already present:", [p for p in PARTS if p not in todo], "| downloading:", todo)
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(3) as ex:
    for msg in ex.map(fetch, todo):
        print(msg)

# official Libri2Mix metadata (source pairs + gains), pinned commit
if not os.path.exists(f"{META_DIR}/metadata/Libri2Mix/libri2mix_test-clean.csv"):
    !git clone -q https://github.com/JorisCos/LibriMix {META_DIR} && git -C {META_DIR} checkout -q b898ca375c03e4039cdb3dbe3dd485aad58ec7d1
!du -sh {LS_DIR}

## Scripts (own implementation; generated from the repo's `part1_separation/` folder)

In [ ]:
%%writefile common.py
"""Shared code for Part 1: data, losses, metrics, and model wrappers.

Own implementation unless noted. The SR-CorrNet network itself (``Model``) and its
STFT/iSTFT layers come from the official repo (github.com/dmlguq456/SR_CorrNet_SS, MIT).
"""
from __future__ import annotations

import argparse
import os
import random
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F

SR = 8000  # the pretrained SR-CorrNet checkpoint operates at 8 kHz
HF_REPO = "shinuh/sr-corrnet-ss-1ch-wsj-fix-2spk"
EPS = 1e-8


# Libri2Mix discovery
def _split_name(path: Path) -> str | None:
    for part in reversed(path.parts):
        p = part.lower()
        if p.startswith("train"):
            return "train"
        if p.startswith("dev") or p in ("valid", "val", "cv"):
            return "dev"
        if p.startswith("test") or p == "tt":
            return "test"
    return None


LIBRIMIX_SPLITS = {"train": "train-clean-100", "dev": "dev-clean", "test": "test-clean"}


def _roots(root: str) -> list[str]:
    return [r for r in str(root).split(",") if r]


def _find_wav_splits(roots: list[str], mix_type: str) -> dict[str, dict]:
    candidates: dict[str, list[tuple[int, Path]]] = {}
    for root in roots:
        for dirpath, dirnames, _ in os.walk(root, followlinks=True):
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and not d.isdigit()]
            names = set(dirnames)
            if {"s1", "s2"} <= names and mix_type in names:
                d = Path(dirpath)
                split = _split_name(d)
                if split is None:
                    continue
                s = str(d).lower()
                score = (2 if "8k" in s else 0) + (1 if "min" in s else 0) + (1 if "100" in s else 0)
                candidates.setdefault(split, []).append((score, d))
    found = {}
    for split, cands in candidates.items():
        cands.sort(key=lambda x: -x[0])
        d = cands[0][1]
        mix_files = {p.name for p in (d / mix_type).glob("*.wav")}
        s1_files = {p.name for p in (d / "s1").glob("*.wav")}
        s2_files = {p.name for p in (d / "s2").glob("*.wav")}
        ids = sorted(mix_files & s1_files & s2_files)
        if not ids:
            continue
        sr = sf.info(str(d / mix_type / ids[0])).samplerate
        found[split] = {"kind": "wav", "dir": str(d), "sr": sr, "ids": ids, "mix_type": mix_type}
    return found


def _find_meta_splits(roots: list[str]) -> dict[str, dict]:
    """Libri2Mix built on the fly from LibriSpeech FLACs + the official LibriMix metadata CSVs.

    Uses ``metadata/Libri2Mix/libri2mix_<split>.csv`` (source paths and gains) exactly as the
    official generator does: sources * gain, truncated to the shorter one ("min" mode), summed.
    Noise columns are ignored (= ``mix_clean``). Speaker sex comes from ``*_info.csv``.
    """
    import pandas as pd
    meta, ls_dirs = None, {}
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
            if meta is None and "libri2mix_test-clean.csv" in filenames:
                meta = Path(dirpath)
            for name in dirnames:
                if name in LIBRIMIX_SPLITS.values() and name not in ls_dirs:
                    cand = Path(dirpath) / name
                    subs = next(os.walk(cand), (None, [], None))[1]
                    if subs and all(x.isdigit() for x in subs[:5]):  # LibriSpeech speaker folders
                        ls_dirs[name] = cand
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and not d.isdigit()]
    found = {}
    if meta is None:
        return found
    for split, ls_name in LIBRIMIX_SPLITS.items():
        csv = meta / f"libri2mix_{ls_name}.csv"
        if not csv.exists() or ls_name not in ls_dirs:
            continue
        base = ls_dirs[ls_name].parent  # metadata paths start with "<ls_name>/spk/chapter/..."
        df = pd.read_csv(csv)
        rows = {r.mixture_ID: (str(base / r.source_1_path), float(r.source_1_gain),
                               str(base / r.source_2_path), float(r.source_2_gain)) for r in df.itertuples()}
        sex = {}
        info_csv = meta / f"libri2mix_{ls_name}_info.csv"
        if info_csv.exists():
            di = pd.read_csv(info_csv)
            sex = {r.mixture_ID: "".join(sorted(str(r.speaker_1_sex) + str(r.speaker_2_sex))) for r in di.itertuples()}
        first = next(iter(rows.values()))[0]
        if not os.path.exists(first):
            raise FileNotFoundError(f"metadata points to {first}, which does not exist")
        found[split] = {"kind": "meta", "dir": str(ls_dirs[ls_name]), "sr": sf.info(first).samplerate,
                        "ids": sorted(rows), "rows": rows, "sex": sex, "mix_type": "mix_clean (on the fly)"}
    return found


def find_libri2mix(root: str, mix_type: str = "mix_clean") -> dict[str, dict]:
    """Return {split: info} for train/dev/test. ``root`` may be several comma-separated dirs.

    1. Generated Libri2Mix wavs: split folders (train-100/dev/test) containing ``<mix_type>/``, ``s1/``,
       ``s2/``; wav8k or wav16k, prefers wav8k and ``min``.
    2. Otherwise raw LibriSpeech (train-clean-100, dev-clean, test-clean) + LibriMix ``metadata/``:
       mixtures are created on the fly, identical to Libri2Mix ``min`` / ``mix_clean``.
    """
    roots = _roots(root)
    return _find_wav_splits(roots, mix_type) or _find_meta_splits(roots)


# Audio I/O
def resample(x: torch.Tensor, sr_in: int, sr_out: int = SR) -> torch.Tensor:
    """Polyphase resampling (scipy), e.g. Libri2Mix wav16k -> 8 kHz. Works on [..., L]."""
    if sr_in == sr_out:
        return x
    from math import gcd
    from scipy.signal import resample_poly
    g = gcd(sr_in, sr_out)
    y = resample_poly(x.numpy(), sr_out // g, sr_in // g, axis=-1)
    return torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))


def load_wav(path: str, start: int = 0, frames: int = -1) -> tuple[torch.Tensor, int]:
    x, sr = sf.read(path, start=start, frames=frames, dtype="float32", always_2d=True)
    return torch.from_numpy(x[:, 0].copy()), sr


def load_triplet(split_info: dict, utt_id: str, start_s: float | None = None,
                 dur_s: float | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (mix [L], sources [2, L]) at 8 kHz. Optional crop given in seconds."""
    sr = split_info["sr"]
    start = 0 if start_s is None else int(start_s * sr)
    frames = -1 if dur_s is None else int(dur_s * sr)
    if split_info["kind"] == "meta":
        p1, g1, p2, g2 = split_info["rows"][utt_id]
        s1, _ = load_wav(p1, start, frames)
        s2, _ = load_wav(p2, start, frames)
        L = min(len(s1), len(s2))
        srcs = resample(torch.stack([s1[:L] * g1, s2[:L] * g2]), sr)
        return srcs.sum(0), srcs
    d, mt = Path(split_info["dir"]), split_info["mix_type"]
    mix, _ = load_wav(str(d / mt / utt_id), start, frames)
    s1, _ = load_wav(str(d / "s1" / utt_id), start, frames)
    s2, _ = load_wav(str(d / "s2" / utt_id), start, frames)
    L = min(len(mix), len(s1), len(s2))
    mix, srcs = mix[:L], torch.stack([s1[:L], s2[:L]])
    mix, srcs = resample(mix, sr), resample(srcs, sr)
    return mix, srcs


def utt_frames(split_info: dict, utt_id: str) -> int:
    """Length in samples at the source rate (shorter source for on-the-fly mixtures)."""
    if split_info["kind"] == "meta":
        p1, _, p2, _ = split_info["rows"][utt_id]
        return min(sf.info(p1).frames, sf.info(p2).frames)
    return sf.info(str(Path(split_info["dir"]) / split_info["mix_type"] / utt_id)).frames


class Libri2MixDataset(torch.utils.data.Dataset):
    """Random fixed-length crops for training, deterministic crops for validation."""

    def __init__(self, split_info: dict, segment_s: float = 4.0, train: bool = True,
                 max_items: int | None = None, seed: int = 0):
        self.info = split_info
        self.ids = list(split_info["ids"])
        if max_items is not None and max_items < len(self.ids):
            rng = random.Random(seed)
            self.ids = sorted(rng.sample(self.ids, max_items))
        self.segment_s = segment_s
        self.train = train
        self.seg = int(segment_s * SR)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        utt = self.ids[i]
        sr = self.info["sr"]
        dur = utt_frames(self.info, utt) / sr
        if dur > self.segment_s:
            start = random.uniform(0, dur - self.segment_s) if self.train else 0.0
            mix, srcs = load_triplet(self.info, utt, start, self.segment_s)
        else:
            mix, srcs = load_triplet(self.info, utt)
        mix, srcs = mix[: self.seg], srcs[:, : self.seg]
        pad = self.seg - mix.shape[-1]
        if pad > 0:  # zero-pad short utterances; zeros are harmless for SI-SDR
            mix = F.pad(mix, (0, pad))
            srcs = F.pad(srcs, (0, pad))
        if self.train:  # random gain: model and loss are scale-invariant, but this de-biases levels
            g = 10 ** (random.uniform(-5, 5) / 20)
            mix, srcs = mix * g, srcs * g
        return mix, srcs


# Losses / metrics (own implementation)
def si_sdr(est: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Scale-invariant SDR in dB over the last axis. Shapes broadcast."""
    est = est - est.mean(-1, keepdim=True)
    ref = ref - ref.mean(-1, keepdim=True)
    alpha = (est * ref).sum(-1, keepdim=True) / (ref.pow(2).sum(-1, keepdim=True) + EPS)
    target = alpha * ref
    noise = est - target
    return 10 * torch.log10(target.pow(2).sum(-1) / (noise.pow(2).sum(-1) + EPS) + EPS)


def pit_si_sdr(est: torch.Tensor, ref: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Utterance-level PIT for 2 sources. est, ref: [B, 2, L].

    Returns (per-source SI-SDR of the best permutation [B, 2], swapped flag [B]).
    """
    keep = si_sdr(est, ref)                       # [B, 2]
    swap = si_sdr(est.flip(1), ref)               # [B, 2]
    use_swap = swap.mean(-1) > keep.mean(-1)
    best = torch.where(use_swap[:, None], swap, keep)
    return best, use_swap


def apply_perm(est: torch.Tensor, swapped: torch.Tensor) -> torch.Tensor:
    return torch.where(swapped[:, None, None], est.flip(1), est)


def mag_snr_loss(est: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Negative SNR between STFT magnitudes (phase-insensitive). Used for SR-CorrNet's
    intermediate decoder outputs, mirroring the auxiliary loss in the official repo."""
    win = torch.hann_window(512, device=est.device)
    def mag(x):
        X = torch.stft(x.reshape(-1, x.shape[-1]), 512, 256, window=win, return_complex=True)
        return X.abs()
    e, r = mag(est), mag(ref)
    snr = 10 * torch.log10(r.pow(2).sum((-1, -2)) / ((e - r).pow(2).sum((-1, -2)) + EPS) + EPS)
    return -snr.clamp(max=30).mean()


# Models
def _load_hf_files(repo_id: str = HF_REPO) -> tuple[dict, str]:
    import yaml
    from huggingface_hub import hf_hub_download
    cfg_path = hf_hub_download(repo_id=repo_id, filename="config.yaml")
    ckpt_path = hf_hub_download(repo_id=repo_id, filename="model.pt")
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)["config"]
    return cfg, ckpt_path


class SRCorrNetSeparator(nn.Module):
    """Waveform-in / waveform-out wrapper around the official SR-CorrNet ``Model``.

    Mirrors the official engine: mixture is std-normalised, transformed with the repo's
    STFT (128/64 at 8 kHz), fed as stacked real/imag, and the estimated complex STFTs
    are inverted with the repo's iSTFT.
    """

    def __init__(self, cfg: dict):
        super().__init__()
        from sr_corrnet.models.SR_CorrNet_SS.model import Model
        from sr_corrnet.utils.util_stft import STFT, iSTFT
        self.cfg = cfg
        self.net = Model(**cfg["model"])
        # build on CPU so the kernels are registered buffers/params and follow .to(device)
        self.stft = STFT(**cfg["stft"], device="cpu", normalize=True)
        self.istft = iSTFT(**cfg["stft"], device="cpu", normalize=True)

    @classmethod
    def pretrained(cls, repo_id: str = HF_REPO) -> "SRCorrNetSeparator":
        from sr_corrnet.utils.util_engine import _fix_compiled_state_dict, _migrate_state_dict
        cfg, ckpt_path = _load_hf_files(repo_id)
        m = cls(cfg)
        raw = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        sd = raw["model_state_dict"] if "model_state_dict" in raw else raw
        sd = _migrate_state_dict(_fix_compiled_state_dict(sd))
        # the released file also carries thop profiler counters (total_ops/total_params); drop them
        sd = {k: v for k, v in sd.items() if not k.endswith(("total_ops", "total_params"))}
        missing, unexpected = m.net.load_state_dict(sd, strict=False)
        n = len(m.net.state_dict())
        print(f"[SR-CorrNet] loaded {repo_id}: {n - len(missing)}/{n} tensors, "
              f"{len(missing)} missing, {len(unexpected)} unexpected")
        if missing:
            raise RuntimeError(f"Pretrained checkpoint is missing weights: {missing[:5]} ...")
        return m

    def enable_grad_checkpointing(self):
        """Recompute each heavy block in the backward pass instead of storing its activations.
        Training memory drops from ~4.2 GiB per audio-second to roughly one block's worth, for
        ~30% more compute. Outputs and gradients are unchanged."""
        from torch.utils.checkpoint import checkpoint
        mods = [self.net.encoder, self.net.filter_estim, *self.net.enc_block, *self.net.dec_block,
                *self.net.dec_cs, *self.net.filter_estim_aux]
        for mod in mods:
            fwd = mod.forward
            def wrapped(*args, _fwd=fwd, **kw):
                if torch.is_grad_enabled():
                    return checkpoint(_fwd, *args, use_reentrant=False, **kw)
                return _fwd(*args, **kw)
            mod.forward = wrapped
        return self

    def encoder_params(self):
        return list(self.net.encoder.parameters()) + list(self.net.enc_block.parameters())

    def _to_wave(self, outs: list, L: int) -> torch.Tensor:
        waves = []
        for e in outs:  # e: [B, M_o, F, T, 2]
            c = torch.complex(e[..., 0], e[..., 1]).squeeze(1)  # [B, F, T]
            w = self.istft(c, cplx=True)
            w = w.reshape(w.shape[0], -1)
            w = w[:, :L] if w.shape[-1] >= L else F.pad(w, (0, L - w.shape[-1]))
            waves.append(w)
        return torch.stack(waves, 1)  # [B, N, L]

    def forward(self, mix: torch.Tensor, aux: bool = False):
        """mix: [B, L] -> est [B, 2, L] (and list of aux estimates if aux=True)."""
        L = mix.shape[-1]
        x = mix.unsqueeze(1)                                  # [B, 1, L]
        X = self.stft(x, cplx=True)                           # [B, 1, F, T]
        inp = torch.cat([X.real, X.imag], dim=1)              # [B, 2, F, T]
        out, out_aux, _ = self.net(inp, aux_loss=aux)
        est = self._to_wave(out, L)
        if aux:
            return est, [self._to_wave(o, L) for o in out_aux]
        return est


class BLSTMMaskNet(nn.Module):
    """From-scratch baseline (own implementation): STFT -> log-magnitude -> 3-layer BLSTM
    -> 2 sigmoid masks applied to the complex mixture STFT -> iSTFT. Trained end-to-end
    with the same waveform PIT SI-SDR loss as SR-CorrNet."""

    def __init__(self, n_fft: int = 256, hop: int = 64, hidden: int = 256, layers: int = 3):
        super().__init__()
        self.n_fft, self.hop = n_fft, hop
        self.register_buffer("win", torch.hann_window(n_fft), persistent=False)
        F_ = n_fft // 2 + 1
        self.norm = nn.LayerNorm(F_)
        self.rnn = nn.LSTM(F_, hidden, layers, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * hidden, 2 * F_)

    def forward(self, mix: torch.Tensor, aux: bool = False):
        L = mix.shape[-1]
        X = torch.stft(mix, self.n_fft, self.hop, window=self.win, return_complex=True)  # [B, F, T]
        feat = torch.log1p(X.abs()).transpose(1, 2)                                       # [B, T, F]
        h, _ = self.rnn(self.norm(feat))
        m = torch.sigmoid(self.proj(h)).view(h.shape[0], h.shape[1], 2, -1).permute(0, 2, 3, 1)  # [B,2,F,T]
        S = m * X.unsqueeze(1)
        est = torch.istft(S.reshape(-1, *S.shape[2:]), self.n_fft, self.hop, window=self.win, length=L)
        est = est.view(mix.shape[0], 2, L)
        return (est, []) if aux else est


def build_model(name: str, init: str = "pretrained") -> nn.Module:
    if name == "srcorrnet":
        if init == "pretrained":
            return SRCorrNetSeparator.pretrained()
        cfg, _ = _load_hf_files()
        return SRCorrNetSeparator(cfg)
    if name == "baseline":
        return BLSTMMaskNet()
    raise ValueError(name)


def load_model(name: str, ckpt: str) -> nn.Module:
    """ckpt = 'pretrained' (zero-shot SR-CorrNet) or a path saved by train.py."""
    if ckpt == "pretrained":
        return build_model(name, "pretrained")
    model = build_model(name, "scratch")
    state = torch.load(ckpt, map_location="cpu", weights_only=True)
    state = state.get("model", state)
    model.load_state_dict(state)
    return model


# Inference on arbitrary-length audio
@torch.inference_mode()
def separate(model: nn.Module, mix: torch.Tensor, device: torch.device,
             max_full_s: float = 20.0, chunk_s: float = 8.0, overlap_s: float = 2.0) -> torch.Tensor:
    """mix: [L] at 8 kHz -> est [2, L]. Runs the whole file at once when it fits; otherwise
    falls back to overlapping chunks whose output order is aligned to the previous chunk by
    correlation on the overlap (chunk-wise processing reintroduces the permutation problem)."""
    model.eval()
    scale = mix.std() + EPS
    x = (mix / scale).to(device)
    L = x.shape[-1]
    if L <= max_full_s * SR:
        try:
            return model(x[None])[0].cpu() * scale
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    C, O = int(chunk_s * SR), int(overlap_s * SR)
    hop = C - O
    out = torch.zeros(2, L)
    weight = torch.zeros(L)
    fade = torch.ones(C)
    ramp = torch.linspace(0, 1, O)
    fade[:O], fade[-O:] = ramp, ramp.flip(0)
    prev_tail = None
    for start in range(0, max(L - O, 1), hop):
        seg = x[start:start + C]
        n = seg.shape[-1]
        est = model(F.pad(seg, (0, C - n))[None])[0, :, :n].cpu()
        if prev_tail is not None:
            head = est[:, :O]
            keep = (head * prev_tail).sum()
            swap = (head.flip(0) * prev_tail).sum()
            if swap > keep:
                est = est.flip(0)
        w = fade[:n].clone()
        if start == 0:
            w[:O] = 1
        if start + C >= L:
            w[-min(O, n):] = 1
        out[:, start:start + n] += est * w
        weight[start:start + n] += w
        prev_tail = est[:, -O:] if n == C else None
    return out / weight.clamp(min=EPS) * scale


def seed_everything(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Print the Libri2Mix splits found under a root dir.")
    ap.add_argument("--data_root", default="/kaggle/input")
    ap.add_argument("--mix_type", default="mix_clean")
    a = ap.parse_args()
    found = find_libri2mix(a.data_root, a.mix_type)
    if not found:
        raise SystemExit(f"No Libri2Mix split with {a.mix_type}/, s1/, s2/ found under {a.data_root}")
    for k, v in found.items():
        print(f"{k:6s} {len(v['ids']):6d} mixtures  {v['sr']} Hz  {v['mix_type']}  {v['dir']}")

In [ ]:
%%writefile train.py
"""Fine-tune SR-CorrNet (or train the BLSTM baseline) on Libri2Mix.

Own implementation of the training loop. Examples:
  python train.py --model srcorrnet --init pretrained --run_name ft_full
  python train.py --model srcorrnet --init pretrained --freeze_encoder --run_name ft_frozen_enc
  python train.py --model baseline --init scratch --lr 1e-3 --run_name blstm
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from common import (Libri2MixDataset, build_model, find_libri2mix, mag_snr_loss, pit_si_sdr,
                    apply_perm, seed_everything, si_sdr)


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", default="/kaggle/input")
    ap.add_argument("--mix_type", default="mix_clean", help="mix_clean = two speakers, no noise")
    ap.add_argument("--model", choices=["srcorrnet", "baseline"], default="srcorrnet")
    ap.add_argument("--init", choices=["pretrained", "scratch"], default="pretrained")
    ap.add_argument("--freeze_encoder", action="store_true")
    ap.add_argument("--grad_ckpt", action="store_true",
                    help="gradient checkpointing for SR-CorrNet (needed on 16 GB GPUs; same gradients, ~30%% slower)")
    ap.add_argument("--run_name", default="ft_full")
    ap.add_argument("--out_root", default="runs")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--weight_decay", type=float, default=1e-2)
    ap.add_argument("--warmup_steps", type=int, default=300)
    ap.add_argument("--epochs", type=int, default=10, help="ignored when --budget_h is set")
    ap.add_argument("--budget_h", type=float, default=None,
                    help="wall-clock training budget; epochs and the LR schedule are sized to fit it")
    ap.add_argument("--steps_per_epoch", type=int, default=2000, help="optimizer steps per 'epoch'")
    ap.add_argument("--batch_size", type=int, default=2)
    ap.add_argument("--accum", type=int, default=2, help="gradient accumulation steps")
    ap.add_argument("--segment", type=float, default=4.0, help="training crop length in seconds")
    ap.add_argument("--aux_weight", type=float, default=0.5, help="SR-CorrNet decoder aux loss")
    ap.add_argument("--clip", type=float, default=5.0)
    ap.add_argument("--val_items", type=int, default=500)
    ap.add_argument("--val_segment", type=float, default=6.0)
    ap.add_argument("--time_budget_h", type=float, default=11.0, help="hard stop before Kaggle's 12 h limit")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--resume", action="store_true")
    return ap.parse_args()


@torch.no_grad()
def validate(model, loader, device) -> dict:
    model.eval()
    vals, vals_in = [], []
    for mix, srcs in loader:
        mix, srcs = mix.to(device), srcs.to(device)
        scale = mix.std(-1, keepdim=True) + 1e-8
        est = model(mix / scale)
        best, _ = pit_si_sdr(est, srcs)
        vals.append(best.mean(-1).cpu())
        vals_in.append(si_sdr(mix[:, None].expand_as(srcs), srcs).mean(-1).cpu())
    v, vi = torch.cat(vals), torch.cat(vals_in)
    return {"val_si_sdr": v.mean().item(), "val_si_sdri": (v - vi).mean().item()}


def main():
    a = parse()
    seed_everything(a.seed)
    t_start = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out = Path(a.out_root) / a.run_name
    out.mkdir(parents=True, exist_ok=True)
    (out / "args.json").write_text(json.dumps(vars(a), indent=2))
    print("args:", json.dumps(vars(a)))

    splits = find_libri2mix(a.data_root, a.mix_type)
    assert "train" in splits and "dev" in splits, f"need train and dev splits, found {list(splits)}"
    for k, v in splits.items():
        print(f"{k}: {len(v['ids'])} mixtures @ {v['sr']} Hz from {v['dir']}")

    train_ds = Libri2MixDataset(splits["train"], a.segment, train=True)
    val_ds = Libri2MixDataset(splits["dev"], a.val_segment, train=False, max_items=a.val_items)
    train_dl = DataLoader(train_ds, a.batch_size, shuffle=True, num_workers=a.workers,
                          drop_last=True, pin_memory=True, persistent_workers=a.workers > 0)
    val_dl = DataLoader(val_ds, a.batch_size, shuffle=False, num_workers=a.workers)

    model = build_model(a.model, a.init).to(device)
    if a.grad_ckpt:
        assert a.model == "srcorrnet"
        model.enable_grad_checkpointing()
    if a.freeze_encoder:
        assert a.model == "srcorrnet"
        for p in model.encoder_params():
            p.requires_grad_(False)
    params = [p for p in model.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in params)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"params: {n_train/1e6:.2f}M trainable / {n_all/1e6:.2f}M total")

    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=a.weight_decay)
    plan = {"epochs": a.epochs, "total": a.epochs * a.steps_per_epoch}  # resized if --budget_h
    def lr_lambda(step):  # linear warm-up then cosine decay to 1% of peak
        if step < a.warmup_steps:
            return (step + 1) / a.warmup_steps
        p = (step - a.warmup_steps) / max(1, plan["total"] - a.warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    history, start_epoch, step, best = [], 1, 0, -1e9
    if a.resume and (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start_epoch, step, best, history = ck["epoch"] + 1, ck["step"], ck["best"], ck["history"]
        if a.budget_h is not None:  # keep the budget-sized schedule; otherwise --epochs may be extended
            plan.update(ck.get("plan", {}))
        print(f"resumed from epoch {ck['epoch']} (best val SI-SDRi {best:.2f} dB)")
    else:
        # Epoch 0 = the starting point. For --init pretrained this is the zero-shot score and
        # doubles as proof that the pretrained weights loaded correctly.
        t_val = time.time()
        v = validate(model, val_dl, device)
        val_seconds = time.time() - t_val
        history.append({"epoch": 0, "step": 0, "train_loss": float("nan"), **v, "lr": 0.0, "minutes": 0.0})
        print(f"[epoch 0] val SI-SDRi {v['val_si_sdri']:.2f} dB")
        best = v["val_si_sdri"]  # fine-tuning must beat the starting point to replace best.pt
        torch.save({"model": model.state_dict()}, out / "best.pt")

    use_aux = a.model == "srcorrnet" and a.aux_weight > 0
    probe = a.budget_h is not None and start_epoch == 1
    it = iter(train_dl)
    epoch = start_epoch - 1
    while epoch < plan["epochs"]:
        epoch += 1
        model.train()
        if a.freeze_encoder:  # keep frozen blocks in eval mode (no dropout/statistics drift)
            model.net.encoder.eval(); model.net.enc_block.eval()
        tot, n = 0.0, 0
        pbar = tqdm(range(a.steps_per_epoch), desc=f"epoch {epoch}", mininterval=30)
        for i_step in pbar:
            if probe and i_step == 5:
                t_probe = time.time()
            if probe and i_step == 35:  # measure speed, then size epochs + cosine to the budget
                step_s = (time.time() - t_probe) / 30
                per_epoch = a.steps_per_epoch * step_s + val_seconds
                remaining = a.budget_h * 3600 - (t_probe - t_start) + 5 * step_s
                plan["epochs"] = max(1, int(remaining // per_epoch))
                plan["total"] = plan["epochs"] * a.steps_per_epoch
                probe = False
                print(f"\n[budget] {step_s:.2f} s/step, {per_epoch/60:.1f} min/epoch -> "
                      f"{plan['epochs']} epochs = {plan['total']} steps in {a.budget_h} h")
            opt.zero_grad(set_to_none=True)
            for _ in range(a.accum):
                try:
                    mix, srcs = next(it)
                except StopIteration:
                    it = iter(train_dl); mix, srcs = next(it)
                mix, srcs = mix.to(device, non_blocking=True), srcs.to(device, non_blocking=True)
                scale = mix.std(-1, keepdim=True) + 1e-8      # same normalisation as the official engine
                mix, srcs = mix / scale, srcs / scale[..., None]
                est, est_aux = model(mix, aux=True)
                best_sdr, swapped = pit_si_sdr(est, srcs)
                loss = -best_sdr.mean()
                if use_aux and est_aux:
                    # aux outputs use the permutation chosen for the final output, as in the repo
                    l_aux = sum(mag_snr_loss(apply_perm(e, swapped), srcs) for e in est_aux) / len(est_aux)
                    loss = loss + a.aux_weight * l_aux
                (loss / a.accum).backward()
                tot += -best_sdr.mean().item(); n += 1
            torch.nn.utils.clip_grad_norm_(params, a.clip)
            opt.step(); sched.step(); step += 1
            pbar.set_postfix(train_si_sdr=f"{-tot/n:.2f}", lr=f"{sched.get_last_lr()[0]:.1e}")

        v = validate(model, val_dl, device)
        minutes = (time.time() - t_start) / 60
        row = {"epoch": epoch, "step": step, "train_loss": tot / n, **v,
               "lr": sched.get_last_lr()[0], "minutes": minutes}
        history.append(row)
        mem = f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB" if device.type == "cuda" else ""
        print(f"[epoch {epoch}] train SI-SDR {-tot/n:.2f} dB | val SI-SDRi {v['val_si_sdri']:.2f} dB | {minutes:.0f} min{mem}")
        if v["val_si_sdri"] > best:
            best = v["val_si_sdri"]
            torch.save({"model": model.state_dict()}, out / "best.pt")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "epoch": epoch, "step": step, "best": best, "history": history, "plan": plan},
                   out / "last.pt")
        with open(out / "history.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            w.writeheader(); w.writerows(history)
        per_epoch_h = (time.time() - t_start) / 3600 / max(1, epoch - start_epoch + 1)
        if (time.time() - t_start) / 3600 + per_epoch_h > a.time_budget_h:
            print(f"stopping: next epoch would exceed the {a.time_budget_h} h budget")
            break

    print(f"done. best val SI-SDRi {best:.2f} dB -> {out/'best.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluate.py
"""Evaluate a separator on full-length Libri2Mix utterances (own implementation).

Writes per-utterance metrics + diagnostics to <out>/per_utt.csv and a summary to
<out>/summary.json. With --save_audio, also writes mixture / references / estimates.

  python evaluate.py --model srcorrnet --ckpt pretrained --out results/zeroshot
  python evaluate.py --model srcorrnet --ckpt runs/ft_full/best.pt --out results/ft_full --sdr --perceptual
  python evaluate.py --model srcorrnet --ckpt runs/ft_full/best.pt --out results/ft_full \
      --ids_file results/example_ids.txt --save_audio
"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm import tqdm

from common import SR, find_libri2mix, load_model, load_triplet, pit_si_sdr, separate, si_sdr


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", default="/kaggle/input")
    ap.add_argument("--mix_type", default="mix_clean")
    ap.add_argument("--split", default="test")
    ap.add_argument("--model", choices=["srcorrnet", "baseline"], default="srcorrnet")
    ap.add_argument("--ckpt", default="pretrained")
    ap.add_argument("--out", required=True)
    ap.add_argument("--max_utts", type=int, default=None, help="evaluate a fixed random subset of N ids (seed 0)")
    ap.add_argument("--ids_file", default=None, help="only evaluate these ids (one per line)")
    ap.add_argument("--save_audio", action="store_true")
    ap.add_argument("--sdr", action="store_true", help="also BSS-eval SDR via mir_eval (slower)")
    ap.add_argument("--perceptual", action="store_true", help="also STOI and narrow-band PESQ")
    return ap.parse_args()


# Diagnostics
def active_mask(x: np.ndarray, frame: int = 256, thr_db: float = -30.0) -> np.ndarray:
    """Frame-level activity: energy within thr_db of the utterance's loudest frame."""
    n = len(x) // frame
    e = (x[: n * frame].reshape(n, frame) ** 2).mean(1) + 1e-12
    return 10 * np.log10(e / e.max()) > thr_db


def median_f0(x: np.ndarray) -> float:
    import librosa
    f0 = librosa.yin(x, fmin=60, fmax=400, sr=SR, frame_length=512, hop_length=256)
    act = active_mask(x, 256)
    m = min(len(f0), len(act))
    v = f0[:m][act[:m]]
    return float(np.median(v)) if len(v) else float("nan")


def swap_fraction(est: torch.Tensor, ref: torch.Tensor, win_s: float = 1.0, margin_db: float = 1.0) -> float:
    """Fraction of 1 s windows (where both speakers are active) whose locally best output
    assignment disagrees with the utterance-level PIT assignment by more than margin_db.
    Utterance-level PIT hides these mid-utterance swaps; this exposes them."""
    W = int(win_s * SR)
    n = ref.shape[-1] // W
    if n < 2:
        return float("nan")
    e = est[:, : n * W].reshape(2, n, W).transpose(0, 1)   # [n, 2, W]
    r = ref[:, : n * W].reshape(2, n, W).transpose(0, 1)
    en = r.pow(2).mean(-1)                                  # [n, 2]
    both = (10 * torch.log10(en / en.max(0).values + 1e-12) > -20).all(-1)
    keep = si_sdr(e, r).mean(-1)
    swap = si_sdr(e.flip(1), r).mean(-1)
    disagree = (swap - keep) > margin_db
    if both.sum() == 0:
        return float("nan")
    return (disagree & both).sum().item() / both.sum().item()


def bss_sdr(est: np.ndarray, ref: np.ndarray) -> float:
    from mir_eval.separation import bss_eval_sources
    sdr, _, _, _ = bss_eval_sources(ref, est, compute_permutation=False)
    return float(sdr.mean())


def perceptual(est: np.ndarray, ref: np.ndarray) -> tuple[float, float]:
    from pystoi import stoi
    from pesq import pesq
    st = float(np.mean([stoi(ref[i], est[i], SR, extended=False) for i in range(2)]))
    try:
        pq = float(np.mean([pesq(SR, ref[i], est[i], "nb") for i in range(2)]))
    except Exception:  # PESQ raises on e.g. no detected speech
        pq = float("nan")
    return st, pq


def to_listening_level(est: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Rescale each estimate by its SI-SDR projection gain so it is level-matched to its
    reference (scale-invariant models output arbitrary gain)."""
    a = (est * ref).sum(-1, keepdim=True) / (est.pow(2).sum(-1, keepdim=True) + 1e-8)
    return est * a


def main():
    a = parse()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    info = find_libri2mix(a.data_root, a.mix_type)[a.split]
    ids = info["ids"]
    if a.ids_file:
        wanted = [l.strip() for l in open(a.ids_file) if l.strip()]
        ids = [i for i in wanted if i in set(ids)]
    elif a.max_utts:
        ids = sorted(random.Random(0).sample(ids, min(a.max_utts, len(ids))))  # same subset for every system
    print(f"{a.split}: evaluating {len(ids)} utterances from {info['dir']} ({info['sr']} Hz -> {SR} Hz)")

    model = load_model(a.model, a.ckpt).to(device).eval()
    rows = []
    for utt in tqdm(ids, mininterval=30):
        mix, ref = load_triplet(info, utt)
        t0 = time.time()
        est = separate(model, mix, device)
        rtf = (time.time() - t0) / (mix.shape[-1] / SR)
        best, swapped = pit_si_sdr(est[None], ref[None])
        est = est.flip(0) if swapped.item() else est          # now est[i] matches ref[i]
        sin = si_sdr(mix[None].expand_as(ref), ref)
        refn, estn, mixn = ref.numpy(), est.numpy(), mix.numpy()
        row = {
            "id": utt,
            "dur_s": mix.shape[-1] / SR,
            "si_sdr_in": sin.mean().item(),
            "si_sdr": best.mean().item(),
            "si_sdri": (best[0] - sin).mean().item(),
            "si_sdr_s1": best[0, 0].item(),
            "si_sdr_s2": best[0, 1].item(),
            "src_level_diff_db": 10 * np.log10((refn[0] ** 2).sum() / ((refn[1] ** 2).sum() + 1e-12) + 1e-12),
            "f0_s1": median_f0(refn[0]),
            "f0_s2": median_f0(refn[1]),
            "overlap": float((active_mask(refn[0]) & active_mask(refn[1])).mean()),
            "swap_frac": swap_fraction(est, ref),
            "rtf": rtf,
        }
        row["df0"] = abs(row["f0_s1"] - row["f0_s2"])
        if info.get("sex"):
            row["sex_pair"] = info["sex"].get(utt, "")  # "FF", "FM" or "MM" from LibriMix metadata
        if a.sdr:
            row["sdr"] = bss_sdr(estn, refn)
            row["sdri"] = row["sdr"] - bss_sdr(np.stack([mixn, mixn]), refn)
        if a.perceptual:
            row["stoi"], row["pesq"] = perceptual(estn, refn)
            row["stoi_in"], row["pesq_in"] = perceptual(np.stack([mixn, mixn]), refn)
        rows.append(row)

        if a.save_audio:
            d = out / "audio" / Path(utt).stem
            d.mkdir(parents=True, exist_ok=True)
            lis = to_listening_level(est, ref)
            peak = max(mix.abs().max(), ref.abs().max(), lis.abs().max()).item() / 0.95
            peak = max(peak, 1.0)
            sf.write(d / "mix.wav", mixn / peak, SR)
            for i in range(2):
                sf.write(d / f"s{i+1}_ref.wav", refn[i] / peak, SR)
                sf.write(d / f"s{i+1}_est.wav", lis[i].numpy() / peak, SR)

    df = pd.DataFrame(rows)
    name = "per_utt_examples.csv" if a.ids_file else "per_utt.csv"
    df.to_csv(out / name, index=False)
    if a.ids_file:
        print(f"saved {len(df)} examples to {out/'audio'}")
        return
    num = df.select_dtypes("number")
    summary = {"model": a.model, "ckpt": a.ckpt, "split": a.split, "n": len(df), "sr": SR,
               "mean": num.mean().round(3).to_dict(), "std": num.std().round(3).to_dict(),
               "median": num.median().round(3).to_dict(),
               "frac_si_sdri_below_5db": float((df.si_sdri < 5).mean()),
               "frac_swap_gt_0": float((df.swap_frac > 0).mean())}
    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    print(f"SI-SDRi {df.si_sdri.mean():.2f} ± {df.si_sdri.std():.2f} dB (median {df.si_sdri.median():.2f}) "
          f"over {len(df)} utts" + (f" | SDRi {df.sdri.mean():.2f}" if a.sdr else "")
          + (f" | STOI {df.stoi.mean():.3f} PESQ {df.pesq.mean():.2f}" if a.perceptual else ""))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze.py
"""Compare evaluated systems and produce report tables / figures (own implementation).

  # 1) choose qualitative + failure examples from the per-utterance CSVs
  python analyze.py select --runs zeroshot=results/zeroshot ft_full=results/ft_full \
      --main ft_full --ref zeroshot --out results/report
  # 2) after re-running evaluate.py --ids_file ... --save_audio, build tables and figures
  python analyze.py report --runs zeroshot=results/zeroshot ft_full=results/ft_full \
      --histories ft_full=runs/ft_full/history.csv --main ft_full --ref zeroshot --out results/report
"""
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

# Fixed categorical order (validated default palette); colour follows the system, not its rank.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "legend.frameon": False,
})


def kv(items):
    return dict(i.split("=", 1) for i in items or [])


def load_runs(runs: dict) -> dict[str, pd.DataFrame]:
    return {k: pd.read_csv(Path(v) / "per_utt.csv") for k, v in runs.items()}


# Example selection
def select(a):
    dfs = load_runs(kv(a.runs))
    m = dfs[a.main].set_index("id")
    picks = []
    def add(i, why):
        if i not in [p[0] for p in picks]:
            picks.append((i, why))
    s = m.si_sdri.sort_values()
    add(s.index[-1], "best")
    add(s.index[len(s) // 2], "median")
    add(s.index[0], "worst")
    add(s.index[1], "2nd worst")
    sw = m[m.swap_frac.notna()].swap_frac.sort_values()
    if len(sw) and sw.iloc[-1] > 0:
        add(sw.index[-1], "highest mid-utterance swap rate")
    add(m.df0.idxmin(), "smallest pitch gap between speakers")
    add(m.src_level_diff_db.abs().idxmax(), "largest level difference between speakers")
    if a.ref and a.ref in dfs:
        r = dfs[a.ref].set_index("id")
        gain = (m.si_sdri - r.si_sdri.reindex(m.index)).dropna().sort_values()
        add(gain.index[-1], f"largest gain over {a.ref}")
        add(gain.index[0], f"largest loss vs {a.ref}")
    if "sex_pair" in m:
        same = m[m.sex_pair.isin(["FF", "MM"])]
        if len(same):
            add(same.si_sdri.idxmin(), "worst same-sex mixture")
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    (out / "example_ids.txt").write_text("\n".join(i for i, _ in picks) + "\n")
    pd.DataFrame(picks, columns=["id", "reason"]).merge(
        m.reset_index(), on="id").to_csv(out / "example_ids.csv", index=False)
    for i, why in picks:
        print(f"{why:45s} {i}  SI-SDRi={m.loc[i, 'si_sdri']:.2f}")


# Report: tables and figures
def fmt(x, d=2):
    return "–" if pd.isna(x) else f"{x:.{d}f}"


def summary_table(dfs) -> str:
    cols = [("si_sdri", "SI-SDRi (dB)"), ("sdri", "SDRi (dB)"), ("stoi", "STOI"), ("pesq", "PESQ-NB")]
    lines = ["| System | N | " + " | ".join(c[1] for c in cols) + " | median SI-SDRi | % utts < 5 dB | % utts with a swap |",
             "|---|---|" + "---|" * len(cols) + "---|---|---|"]
    first = next(iter(dfs.values()))
    if "stoi_in" in first:
        lines.append(f"| Mixture (input) | {len(first)} | 0.00 | 0.00 | {fmt(first.stoi_in.mean(),3)} | "
                     f"{fmt(first.pesq_in.mean())} | – | – | – |")
    for k, d in dfs.items():
        cells = []
        for c, _ in cols:
            if c in d:
                cells.append(f"{fmt(d[c].mean(), 3 if c=='stoi' else 2)} ± {fmt(d[c].std(), 3 if c=='stoi' else 2)}")
            else:
                cells.append("–")
        lines.append(f"| {k} | {len(d)} | " + " | ".join(cells) +
                     f" | {fmt(d.si_sdri.median())} | {100*(d.si_sdri<5).mean():.1f} | {100*(d.swap_frac>0).mean():.1f} |")
    return "\n".join(lines)


def stratified_table(dfs, col, bins, labels, title) -> tuple[str, pd.DataFrame]:
    rows = {}
    for k, d in dfs.items():
        b = pd.cut(d[col], bins=bins, labels=labels, include_lowest=True)
        rows[k] = d.groupby(b, observed=False).si_sdri.mean()
    counts = pd.cut(next(iter(dfs.values()))[col], bins=bins, labels=labels, include_lowest=True).value_counts().reindex(labels)
    t = pd.DataFrame(rows)
    lines = [f"**{title}**", "", "| bin | N | " + " | ".join(t.columns) + " |", "|---|---|" + "---|" * len(t.columns)]
    for lab in labels:
        lines.append(f"| {lab} | {counts[lab]} | " + " | ".join(fmt(t.loc[lab, c]) for c in t.columns) + " |")
    return "\n".join(lines), t


def grouped_bars(t: pd.DataFrame, xlabel: str, path: Path, title: str):
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    n = len(t.columns); w = 0.8 / n; x = np.arange(len(t.index))
    for j, c in enumerate(t.columns):
        ax.bar(x + (j - (n - 1) / 2) * w, t[c].values, w * 0.92, color=SERIES[j], label=c, zorder=3)
    ax.set_xticks(x, t.index.astype(str)); ax.set_xlabel(xlabel); ax.set_ylabel("mean SI-SDRi (dB)")
    ax.grid(axis="x", visible=False); ax.set_title(title, loc="left", fontsize=10)
    ax.legend(ncol=n, loc="upper left", bbox_to_anchor=(0, -0.22))
    fig.savefig(path); plt.close(fig)


def spectrogram_grid(audio_dir: Path, path: Path, title: str):
    names = ["mix", "s1_ref", "s1_est", "s2_ref", "s2_est"]
    files = [audio_dir / f"{n}.wav" for n in names]
    if not all(f.exists() for f in files):
        return False
    fig, axes = plt.subplots(len(names), 1, figsize=(6.4, 7.0), sharex=True)
    for ax, n, f in zip(axes, names, files):
        x, sr = sf.read(f)
        ax.specgram(x + 1e-9, NFFT=256, Fs=sr, noverlap=192, cmap="magma", vmin=-120, vmax=-20)
        ax.set_ylabel(n, rotation=0, ha="right", va="center"); ax.grid(False)
        ax.set_yticks([0, 2000, 4000]); ax.set_yticklabels(["0", "2k", "4k"])
    axes[-1].set_xlabel("time (s)"); axes[0].set_title(title, loc="left", fontsize=9)
    fig.savefig(path); plt.close(fig)
    return True


def report(a):
    runs = kv(a.runs)
    dfs = load_runs(runs)
    out = Path(a.out); figs = out / "figs"; figs.mkdir(parents=True, exist_ok=True)
    md = ["# Part 1 results", "", "All metrics on full-length test utterances at 8 kHz, utterance-level PIT.", "",
          summary_table(dfs), ""]

    # distribution of SI-SDRi
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    lo = min(d.si_sdri.min() for d in dfs.values()); hi = max(d.si_sdri.max() for d in dfs.values())
    bins = np.linspace(np.floor(lo), np.ceil(hi), 50)
    for j, (k, d) in enumerate(dfs.items()):
        ax.hist(d.si_sdri, bins=bins, histtype="step", lw=2, color=SERIES[j], label=f"{k} (mean {d.si_sdri.mean():.1f})")
    ax.set_xlabel("SI-SDRi per utterance (dB)"); ax.set_ylabel("utterances"); ax.legend(loc="upper left")
    ax.set_title("Distribution of per-utterance SI-SDR improvement", loc="left", fontsize=10)
    fig.savefig(figs / "hist_si_sdri.png"); plt.close(fig)
    md += ["![](figs/hist_si_sdri.png)", ""]

    # stratification
    for col, bins_, labels, xl, fname, title in [
        ("df0", [0, 20, 50, 100, 1e4], ["<20 Hz", "20–50", "50–100", ">100 Hz"],
         "|median F0(s1) − median F0(s2)|", "by_df0.png", "SI-SDRi vs pitch gap between speakers"),
        ("src_level_diff_db", None, ["0–1 dB", "1–2", "2–3", "3–5", ">5 dB"],
         "|level difference between sources|", "by_level.png", "SI-SDRi vs source level difference"),
        ("dur_s", [0, 4, 6, 8, 12, 1e4], ["<4 s", "4–6", "6–8", "8–12", ">12 s"],
         "utterance duration", "by_dur.png", "SI-SDRi vs utterance duration"),
    ]:
        if col == "src_level_diff_db":
            dd = {k: d.assign(src_level_diff_db=d.src_level_diff_db.abs()) for k, d in dfs.items()}
            bins_ = [0, 1, 2, 3, 5, 1e4]
        else:
            dd = dfs
        txt, t = stratified_table(dd, col, bins_, labels, title)
        grouped_bars(t, xl, figs / fname, title)
        md += [txt, "", f"![](figs/{fname})", ""]

    # speaker-sex pairing (available when the data comes from LibriMix metadata)
    if all("sex_pair" in d for d in dfs.values()):
        t = pd.DataFrame({k: d.groupby("sex_pair").si_sdri.mean() for k, d in dfs.items()})
        t = t.reindex([x for x in ["FM", "FF", "MM"] if x in t.index])
        counts = next(iter(dfs.values())).sex_pair.value_counts()
        lines = ["**SI-SDRi by speaker-sex pairing**", "", "| pair | N | " + " | ".join(t.columns) + " |",
                 "|---|---|" + "---|" * len(t.columns)]
        for lab in t.index:
            lines.append(f"| {lab} | {counts.get(lab, 0)} | " + " | ".join(fmt(t.loc[lab, c]) for c in t.columns) + " |")
        grouped_bars(t, "speaker-sex pair (F = female, M = male)", figs / "by_sex_pair.png",
                     "SI-SDRi for mixed-sex vs same-sex mixtures")
        md += ["\n".join(lines), "", "![](figs/by_sex_pair.png)", ""]

    # per-utterance before/after scatter
    if a.ref and a.ref in dfs and a.main in dfs:
        m = dfs[a.main].set_index("id").si_sdri; r = dfs[a.ref].set_index("id").si_sdri.reindex(m.index)
        fig, ax = plt.subplots(figsize=(3.6, 3.6))
        ax.scatter(r, m, s=9, color=SERIES[0], alpha=0.5, lw=0)
        lim = [min(r.min(), m.min()) - 1, max(r.max(), m.max()) + 1]
        ax.plot(lim, lim, color=INK2, lw=1, ls="--"); ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel(f"{a.ref} SI-SDRi (dB)"); ax.set_ylabel(f"{a.main} SI-SDRi (dB)")
        ax.set_title("Per-utterance change", loc="left", fontsize=10)
        fig.savefig(figs / "scatter_main_vs_ref.png"); plt.close(fig)
        better = (m > r).mean()
        md += [f"{a.main} beats {a.ref} on {100*better:.1f}% of utterances "
               f"(mean change {np.nanmean(m - r):+.2f} dB).", "", "![](figs/scatter_main_vs_ref.png)", ""]

    # learning curves
    hist = kv(a.histories)
    if hist:
        fig, ax = plt.subplots(figsize=(6.4, 3.0))
        names = list(dfs.keys())
        for k, p in hist.items():
            h = pd.read_csv(p)
            j = names.index(k) if k in names else len(names) + list(hist).index(k)
            ax.plot(h.epoch, h.val_si_sdri, marker="o", ms=4, color=SERIES[j % len(SERIES)], label=k)
        ax.set_xlabel("epoch (0 = before training)"); ax.set_ylabel("dev SI-SDRi (dB)"); ax.legend()
        ax.set_title("Validation SI-SDRi during training", loc="left", fontsize=10)
        fig.savefig(figs / "learning_curves.png"); plt.close(fig)
        md += ["![](figs/learning_curves.png)", ""]

    # worst utterances with diagnostics
    if a.main in dfs:
        w = dfs[a.main].nsmallest(10, "si_sdri")
        md += [f"**10 worst utterances for {a.main}**", "",
               "| id | SI-SDRi | s1 | s2 | ΔF0 (Hz) | level diff (dB) | overlap | swap frac | dur (s) |",
               "|---|---|---|---|---|---|---|---|---|"]
        for _, r_ in w.iterrows():
            md.append(f"| {r_.id} | {fmt(r_.si_sdri)} | {fmt(r_.si_sdr_s1)} | {fmt(r_.si_sdr_s2)} | {fmt(r_.df0,0)} | "
                      f"{fmt(r_.src_level_diff_db,1)} | {fmt(r_.overlap)} | {fmt(r_.swap_frac)} | {fmt(r_.dur_s,1)} |")
        md.append("")
        corr = dfs[a.main][["si_sdri", "df0", "src_level_diff_db", "overlap", "swap_frac", "dur_s"]].copy()
        corr["src_level_diff_db"] = corr.src_level_diff_db.abs()
        c = corr.corr(method="spearman")["si_sdri"].drop("si_sdri")
        md += ["Spearman correlation of SI-SDRi with each diagnostic (" + a.main + "): " +
               ", ".join(f"{k} {v:+.2f}" for k, v in c.items()), ""]

    # spectrograms of the selected examples
    ex = out / "example_ids.csv"
    if ex.exists():
        md += ["## Qualitative examples", ""]
        for _, e in pd.read_csv(ex).iterrows():
            stem = Path(e.id).stem
            for k in dfs:
                ad = Path(runs[k]) / "audio" / stem
                f = figs / f"spec_{k}_{stem}.png"
                si = dfs[k].set_index("id").si_sdri.get(e.id, np.nan)
                if spectrogram_grid(ad, f, f"{k} | {e.reason} | SI-SDRi {si:.1f} dB"):
                    md += [f"*{e.reason}* ({k}, `{e.id}`): ![]({f.relative_to(out).as_posix()})", ""]
    (out / "results.md").write_text("\n".join(md), encoding="utf-8")
    print(f"wrote {out/'results.md'} and {len(list(figs.glob('*.png')))} figures")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["select", "report"])
    ap.add_argument("--runs", nargs="+", required=True, help="name=results_dir ...")
    ap.add_argument("--histories", nargs="*", default=[], help="name=history.csv ...")
    ap.add_argument("--main", required=True)
    ap.add_argument("--ref", default=None)
    ap.add_argument("--out", default="results/report")
    a = ap.parse_args()
    select(a) if a.cmd == "select" else report(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference.py
"""Separate any mono/stereo wav (or a folder of wavs) into two sources.

  python inference.py --ckpt runs/ft_full/best.pt --input mixture.wav --output separated/
  python inference.py --ckpt pretrained --input some_dir/ --output separated/

Input is resampled to 8 kHz (the model's rate); outputs are written at 8 kHz as
<name>_s1.wav and <name>_s2.wav. Output order is arbitrary (permutation-invariant model).
"""
import argparse
from pathlib import Path

import soundfile as sf
import torch

from common import SR, load_model, load_wav, resample, separate


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["srcorrnet", "baseline"], default="srcorrnet")
    ap.add_argument("--ckpt", default="pretrained")
    ap.add_argument("--input", required=True)
    ap.add_argument("--output", required=True)
    a = ap.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model(a.model, a.ckpt).to(device).eval()
    inp = Path(a.input)
    files = sorted(inp.glob("*.wav")) if inp.is_dir() else [inp]
    out = Path(a.output); out.mkdir(parents=True, exist_ok=True)
    for f in files:
        x, sr = load_wav(str(f))
        x = resample(x, sr, SR)
        est = separate(model, x, device)
        peak = est.abs().max().clamp(min=1e-8)
        est = est / peak * 0.9 if peak > 0.9 else est
        for i in range(2):
            sf.write(out / f"{f.stem}_s{i+1}.wav", est[i].numpy(), SR)
        print(f"{f.name} -> {f.stem}_s1.wav, {f.stem}_s2.wav")


if __name__ == "__main__":
    main()

## 1. Data check

In [ ]:
!python common.py --data_root {DATA_ROOT}

## 2. Fine-tune SR-CorrNet end-to-end (runs first, so a GPU problem shows up within minutes)
Peak LR 1e-4 (5x below the from-scratch recipe) with warm-up and cosine decay sized to the budget,
4 s random crops, batch 1 x accumulation 4, gradient checkpointing, PIT SI-SDR loss + 0.5 x auxiliary decoder loss.
Epoch 0 in the log is the starting point and must match the zero-shot dev score.

In [ ]:
!python train.py --data_root {DATA_ROOT} --model srcorrnet --init pretrained --run_name ft_full --budget_h {FT_HOURS} {FT_BATCH} {STEPS} {RES}

In [ ]:
# stop here if fine-tuning crashed: otherwise best.pt is just the epoch-0 (pretrained) copy
assert os.path.exists("runs/ft_full/history.csv"), "fine-tuning did not finish an epoch, see the error above"
print(open("runs/ft_full/history.csv").read())
!python evaluate.py --data_root {DATA_ROOT} --model srcorrnet --ckpt runs/ft_full/best.pt --out results/ft_full {EVAL_N} {EVAL_EXTRA}

## 3. Zero-shot baseline: the pretrained checkpoint, no training

In [ ]:
!python evaluate.py --data_root {DATA_ROOT} --model srcorrnet --ckpt pretrained --out results/zeroshot {EVAL_N} {EVAL_EXTRA}

## 4. Ablation: freeze the separation encoder, adapt only split, decoder and filter heads

In [ ]:
if RUN_FROZEN:
    !python train.py --data_root {DATA_ROOT} --model srcorrnet --init pretrained --freeze_encoder --run_name ft_frozen --budget_h {FROZEN_HOURS} {FT_BATCH} {STEPS} {RES}
    !python evaluate.py --data_root {DATA_ROOT} --model srcorrnet --ckpt runs/ft_frozen/best.pt --out results/ft_frozen {EVAL_N}

## 5. From-scratch baseline: own BLSTM mask network, same data, loss and evaluation

In [ ]:
if RUN_BLSTM:
    !python train.py --data_root {DATA_ROOT} --model baseline --init scratch --lr 1e-3 --batch_size 8 --accum 1 --warmup_steps 200 --run_name blstm --budget_h {BLSTM_HOURS} {STEPS} {RES}
    !python evaluate.py --data_root {DATA_ROOT} --model baseline --ckpt runs/blstm/best.pt --out results/blstm {EVAL_N}

## 6. Examples, failure cases, tables and figures

In [ ]:
import os
systems = {"zeroshot": ("srcorrnet", "pretrained"), "ft_full": ("srcorrnet", "runs/ft_full/best.pt"),
           "ft_frozen": ("srcorrnet", "runs/ft_frozen/best.pt"), "blstm": ("baseline", "runs/blstm/best.pt")}
systems = {k: v for k, v in systems.items() if os.path.exists(f"results/{k}/per_utt.csv")}
RUNS = " ".join(f"{k}=results/{k}" for k in systems)
HISTS = " ".join(f"{k}=runs/{k}/history.csv" for k in systems if os.path.exists(f"runs/{k}/history.csv"))
!python analyze.py select --runs {RUNS} --main ft_full --ref zeroshot --out results/report
for k, (m, ck) in systems.items():
    !python evaluate.py --data_root {DATA_ROOT} --model {m} --ckpt {ck} --out results/{k} --ids_file results/report/example_ids.txt --save_audio
!python analyze.py report --runs {RUNS} --histories {HISTS} --main ft_full --ref zeroshot --out results/report

In [ ]:
from IPython.display import Markdown, Audio, Image, display
from pathlib import Path
import pandas as pd
for line in open("results/report/results.md", encoding="utf-8").read().split("\n\n"):
    if "![](" in line:  # render figures as images, text and tables as markdown
        display(Markdown(line.split("![](")[0])); display(Image("results/report/" + line.split("![](")[1].rstrip(")")))
    else:
        display(Markdown(line))
ex = pd.read_csv("results/report/example_ids.csv")
for _, e in ex.head(4).iterrows():
    stem = Path(e.id).stem
    display(Markdown(f"**{e.reason}**: `{e.id}`"))
    for f in ["mix", "s1_ref", "s1_est", "s2_ref", "s2_est"]:
        print(f); display(Audio(f"results/ft_full/audio/{stem}/{f}.wav"))

In [ ]:
# everything you need for the report, without the large checkpoints
!zip -qr part1_results.zip results runs/*/history.csv runs/*/args.json && ls -lh part1_results.zip